<a href="https://colab.research.google.com/github/saicharan86/AI-Agents/blob/main/Project_1_HBR_Apple_RAG_Assignment_SaiAmujala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement

## Business Context

As organizations grow and scale, they are often inundated with large volumes of data, reports, and documents that contain critical information for decision-making. In real-world business settings, such as venture capital firms like Andreesen Horowitz, business analysts are required to sift through large datasets, research papers, or reports to extract relevant information that impacts strategic decisions.

For instance, consider that you've just joined Andreesen Horowitz, a renowned venture capital firm, and you are tasked with analyzing a dense report like the Harvard Business Review's **"How Apple is Organized for Innovation."** Going through the report manually can be extremely time-consuming as the size and complexity of these report increases. However, by using **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models, you can significantly streamline this process.

Imagine having the capability to directly ask questions like, “How does Apple structure its teams for innovation?” and get immediate, relevant answers drawn from the report. This ability to extract and organize specific insights quickly and accurately enables you to focus on higher-level analysis and decision-making, rather than being bogged down by information retrieval.

## Objective

The goal is to develop a RAG application that helps business analysts efficiently extract key insights from extensive reports, such as “How Apple is Organized for Innovation.”

Specifically, the system aims to:

- Answer user queries by retrieving relevant content directly from lengthy documents.

- Support natural-language interaction without requiring a full manual read-through.

- Act as an intelligent assistant that streamlines the report analysis process.

Through this solution, analysts can save time, improve productivity, and make faster, more informed strategic decisions

## Data Description

“How Apple is Organized for Innovation” is a detailed Harvard Business Review article that examines Apple’s unique approach to structuring teams, driving innovation, and maintaining a culture of excellence. The article is provided as a PDF consisting of 11 pages, offering in-depth insights into Apple’s organizational design, leadership principles, and decision-making processes.

## Installing and Importing Necessary Libraries and Dependencies

In [6]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain-openai==0.3.30 \
              langchain-chroma

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

## Imports and environment setup

In [17]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import PyMuPDFLoader                            # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing

from langchain_openai import OpenAIEmbeddings                                   # Create vector embeddings using OpenAI's models
from langchain.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore


from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

#### Loading the model

In [3]:
# Load the JSON file and extract values
file_name = 'config.json'                                                       # Name of the configuration file
with open(file_name, 'r') as file:                                              # Open the config file in read mode
    config = json.load(file)                                                    # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")                                      # Extract the API key from the config
    OPENAI_API_BASE = config.get("OPENAI_API_BASE")                             # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ['OPENAI_API_KEY'] = API_KEY                                          # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = OPENAI_API_BASE                                 # Set API base URL as environment variable

# Initialize OpenAI client
client = OpenAI()

In [4]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use
        messages=[
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article?

In [5]:
question_1 = "Who are the authors of this article and who published this article?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"I would need more specific information about the article you're referring to in order to provide details about the authors and the publisher. If you can share the title or any other details, I’d be happy to help!"

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [6]:
question_2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
base_prompt_response_2 = response(question_2)
base_prompt_response_2

'- **Vision**: A strong leader possesses a clear and compelling vision for the future, inspiring others to work towards shared goals. This ability to see the bigger picture fosters motivation and alignment within the team.\n\n- **Empathy**: Effective leaders demonstrate empathy by understanding and acknowledging the feelings and perspectives of their team members. This builds trust and rapport, creating a supportive environment that enhances collaboration and productivity.\n\n- **Decisiveness**: A good leader makes informed decisions promptly, balancing analysis with intuition. This decisiveness instills confidence in the team, enabling swift action and adaptability in a fast-paced environment.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [7]:
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

'While I don\'t have access to specific articles, I can provide insights based on well-known examples of Apple\'s leadership approach that have led to successful innovations. Apple, under the leadership of figures like Steve Jobs and Tim Cook, has consistently demonstrated a few key principles that have driven their innovative success:\n\n1. **Focus on Design and User Experience**: Apple\'s commitment to design excellence is evident in products like the iPhone and MacBook. For instance, the introduction of the iPhone in 2007 revolutionized the smartphone industry. Apple\'s leadership emphasized a sleek, user-friendly interface combined with powerful functionality, setting a new standard for mobile devices.\n\n2. **Integration of Hardware and Software**: Apple\'s approach to creating a seamless ecosystem where hardware and software work harmoniously has resulted in successful innovations like the iPad and Apple Watch. By controlling both aspects, Apple ensures that the user experience i

## Observations:

** Question 1**:

The model fails to answer question 1 because it requires document-specific knowledge, which is not in its immediate context. It defaults to a fallback response, indicating lack of access rather than attempting an answer.


** Question 2**:
The model gives an answer, but it’s generic and not based on the actual article. It uses general knowledge from training instead of information from the document.


** Question 3**:

The model recognizes its limitation but still gives a general answer. However, the response isn’t based on the article, even though the question specifically requires it.



## Question Answering using LLM with Prompt Engineering

Defining a system prompt that aligns with the business problem

In [8]:
system_prompt = """
You are an expert Business Analysis Assistant. Your primary role is to help business analysts extract, interpret, and synthesize insights from business reports and documents
based on LLM training data.

Follow these guidelines when answering:
- When Answering, prioritize factual correctness, align with widely accepted standards, and ensure clarity for  users.
- If a query requires specific reference materials, Reference the source contrnt explicitly rather than speculating.
- If the query is vague, infer the most likely business intent and answer accordingly.
- Summarize key points
- Provide structured, business-friendly explanations
- Keep the tone and style professional, analytical and concise

"""

Defining the function to Generate a Response From the LLM

In [9]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",                                                    # Specify the model to use (GPT-4o in this case)
        messages=[
            {"role": "system", "content": system_prompt},                       # System prompt sets the assistant's behavior
            {"role": "user", "content": user_prompt}                            # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,                                                  # Max number of tokens to generate in the response
        temperature=temperature,                                                # Controls randomness in output (0 = deterministic)
        top_p=top_p                                                             # Controls diversity via nucleus sampling
    )
    return completion.choices[0].message.content                                # Return the text content from the model's reply

### Question 1: Who are the authors of this article and who published this article?

In [10]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

'To assist you effectively, please provide the title of the article or any specific details about it. This will allow me to help you identify the authors and the publisher.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [11]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

'Here are three key leadership characteristics:\n\n- **Vision**: Effective leaders possess a clear and compelling vision for the future, which inspires and motivates their team towards common goals.\n\n- **Communication**: Strong leaders excel in communication, articulating ideas and expectations clearly, and fostering an open environment for feedback and collaboration.\n\n- **Integrity**: Leaders with integrity demonstrate honesty and ethical behavior, building trust and credibility with their team, which is essential for fostering a positive organizational culture.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [12]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"To provide a detailed analysis of Apple's approach to leadership and how it has led to successful innovations, I would need to reference specific content from an article. However, I can summarize widely recognized examples based on established knowledge about Apple's leadership strategies.\n\n### Key Examples of Apple's Leadership and Innovation\n\n1. **Visionary Leadership of Steve Jobs**:\n   - **Product Focus**: Under Steve Jobs, Apple prioritized user experience and design. The introduction of the iPod in 2001 revolutionized the music industry by emphasizing simplicity and aesthetics. Jobs's vision of a seamless integration of hardware and software paved the way for innovative products.\n  \n2. **The iPhone Launch**:\n   - **Cross-Functional Collaboration**: Tim Cook's leadership style, characterized by operational excellence, facilitated the development of the iPhone in 2007. The product combined various technologies (camera, phone, internet) into a single device, showcasing Appl

## Observations

** Question 1**:

The model follows the system prompt strictly and avoids guessing since no source context is provided. It asks for more information instead of hallucinating, showing improved factual discipline compared to the no-prompt version. However, it still cannot answer because it lacks the context or specific artical knowledge.

** Question 2**:

The model provides a structured and professional answer as instructed by the system prompt. However, the response is still generic and not based on the actual article, indicating reliance on general knowledge and traning data of LLM. The prompt improved the style, but not the substance. The answer looks right but isn't based on the actual text.

** Question 3**:

The model plays it safe: it admits it’s missing the specific details but tries to help anyway. By mixing a "heads up" with some generic examples, it ends up sounding relevant without actually using the article.

## Data Preparation for RAG

### Loading the Data

In [18]:
# Set the path to the PDF file
manual_pdf_path = "/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf"      # Path to the HBR Apple PDF

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(manual_pdf_path)                                     # Initialize the PDF loader with the file path

# Extract content from the PDF
document = pdf_loader.load()                                                    # Load and extract text from all pages of the PDF

### Data Overview

In [19]:
print(f"Loaded {len(document)} pages from the PDF.")
print("First page preview:\n")
print(document[0].page_content[:1200])

Loaded 11 pages from the PDF.
First page preview:

REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.


### Data Chunking

In [20]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',                                                # Encoding used by popular LLMs
    chunk_size=800,                                                             # Each chunk will have up to 800 tokens
    chunk_overlap=100                                                           # 100 tokens will overlap between consecutive chunks (for context continuity)
)

document_chunks = text_splitter.split_documents(document)
print(f"Created {len(document_chunks)} chunks.")
print("Example chunk:\n")
print(document_chunks[0].page_content[:1200])

Created 17 chunks.
Example chunk:

REPRINT R2006F
PUBLISHED IN HBR
NOVEMBER–DECEMBER 2020
ARTICLE
ORGANIZATIONAL CULTURE
How Apple Is 
Organized  
for Innovation
It’s about experts leading experts. 
by Joel M. Podolny and Morten T. Hansen
This article is made available to you with compliments of Apple Inc for your personal use. Further posting, copying or distribution is not permitted.


### Embedding

In [21]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,                                                     # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE                                             # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


In [22]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)
print(len(embedding_1))
print(len(embedding_2))

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

1536
1536


([0.002524683950468898,
  -0.005738068372011185,
  -0.003138885833323002,
  -0.01189995277673006,
  0.01745591312646866,
  0.030461760237812996,
  -0.010211312212049961,
  0.01806514896452427,
  -0.013694548048079014,
  -0.021495407447218895,
  0.0029650551732629538,
  0.0092577263712883,
  -0.00790681317448616,
  -0.006651921197772026,
  -0.033613890409469604,
  0.019839877262711525,
  0.02409127913415432,
  -0.012893271632492542,
  -0.0008468038286082447,
  -0.0019022044725716114,
  -0.033613890409469604,
  -0.0015065327752381563,
  -0.004450066015124321,
  0.009171638637781143,
  0.008939864113926888,
  0.00047472334699705243,
  0.0159725584089756,
  -0.02084643952548504,
  0.029243290424346924,
  -0.005797667894512415,
  0.019601481035351753,
  0.017204273492097855,
  -0.011701289564371109,
  0.0012731029419228435,
  -0.009847095236182213,
  -0.02574680931866169,
  -0.0079862792044878,
  -0.023097960278391838,
  0.02491242252290249,
  -0.0384083054959774,
  0.04071280360221863,
  0

### Vector Database

Setup Vector Store Directory

In [23]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = 'research_db'                                                         # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)                                                        # Make directory to save vector store files

Create Vector Store From Documents

In [24]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

Load Vector Store


In [26]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,                                                  # Load existing vector DB files
    embedding_function=embedding_model                                          # Use the same embedding function for queries
)

Exploring Vector store

In [27]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x79bf50c26000>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x79bf50c25250>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [29]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
   # "Which specific management action did Steve jobs take in his first year back to unify Apple's departments?",
   "who are the authors of the article?",
    k=3
)

[Document(metadata={'source': '/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'subject': '', 'format': 'PDF 1.6', 'producer': 'Adobe PDF Library 15.0 (via http://bfo.com/products/pdf?version=2.23.5-r33279)', 'author': '', 'creationDate': "D:20201005141842-04'00'", 'moddate': '2020-12-01T18:37:49+00:00', 'page': 2, 'creationdate': '2020-10-05T14:18:42-04:00', 'trapped': '', 'keywords': '', 'total_pages': 11, 'title': '', 'creator': 'Adobe InDesign 14.0 (Macintosh)', 'file_path': '/content/HBR_How_Apple_Is_Organized_For_Innovation.pdf', 'encryption': 'Standard V2 R3 128-bit RC4', 'modDate': 'D:20201201183749Z'}, page_content='PHOTOGRAPHER\u2002MIKAEL JANSSON\nHow Apple Is \nOrganized \nfor Innovation\nIt’s about experts \nleading experts.\nORGANIZATIONAL \nCULTURE\nJoel M. \nPodolny\nDean, Apple \nUniversity\nMorten T. \nHansen\nFaculty, Apple \nUniversity\nAUTHORS\nFOR ARTICLE REPRINTS CALL 800-988-0886 OR 617-783-7500, OR VISIT HBR.ORG\nHarvard Business Review\nNovember–Decem

### Retriever

Convert Vector Store into a Retriever and Retrieve Relevant Documents

In [30]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type='similarity',                                                   # Use similarity search (based on vector distance)
    search_kwargs={'k': 3}                                                      # Retrieve top 3 most relevant documents
)

#### System and User Prompt Template

In [31]:
# Define the system prompt for the model
qna_system_message = """
You are a Business Analysis Assistant.
Use only the context below to answer the user question.
If the answer is not in the context, say: "I could not find that in the report."

Context:
{context}

Question:
{question}

Answer in clear, simple business English.
"""

In [32]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from the article that are relevant to the question mentioned below:
{context}

###Question
{question}
"""

### Response Function

In [35]:
def generate_rag_response(user_input,k=3,max_tokens=1000,temperature=0.75,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks by passing the user_input to invoke()
    relevant_document_chunks = retriever.invoke(user_input)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article?

In [36]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

'The authors of the article are Joel M. Podolny and Morten T. Hansen. It was published in the Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [39]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

'- **Deep Expertise**: Leaders at Apple are expected to have specialized knowledge in their fields, allowing them to engage meaningfully with their teams and make informed decisions.\n\n- **Immersion in the Details**: Leaders should understand the intricacies of their organization at a granular level, which facilitates quick and effective decision-making across functions.\n\n- **Willingness to Collaboratively Debate**: Apple leaders are encouraged to engage in discussions with various specialists, fostering an environment where different perspectives are valued to enhance decision-making.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [40]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

"One specific example from the article is Roger Rosner, who heads Apple's software application business. His expertise in electrical engineering and extensive experience within Apple allowed him to lead teams that developed successful work-productivity apps like Pages, Numbers, and Keynote. This showcases how having leaders with deep expertise can drive innovation in product development.\n\nAnother example is the team of over 600 experts in camera hardware technology led by Graham Townsend. Their collaboration within a functional organization allows for the pooling of knowledge and expertise, which enhances problem-solving and innovation in camera technology used across various Apple products like iPhones and iPads.\n\nAdditionally, Apple's emphasis on leaders being knowledgeable about details—such as the precise design of product corners—demonstrates how such attention to detail leads to innovative product design, resulting in unique features that differentiate Apple's offerings in th

## Actionable Insights and Business Recommendations

###Observations:

Question 1:

The model correctly retrieves and provides the exact authors and publisher from the document. The response is precise and factual, showing effective use of retrieved context. This demonstrates how RAG enables accurate answers to document-specific queries.

Question 2:

The model provides structured, relevant leadership characteristics directly aligned with the article. The answer is more specific and grounded compared to the generic response without RAG. This shows improved contextual accuracy and relevance.

Question 3:

The model gives concrete examples from the article, including names and real scenarios. The response is detailed and clearly grounded in the source content rather than general knowledge. This highlights RAG’s ability to generate insightful, evidence-based explanations.

## Actionable Insights:
- One key insight from this project is System prompts improve the behavior, but not the knowledge. It helps with the structure and tone but the answer is not factual or grouded.
- RAG significantly improves accuracy, relevance, and grounding of responses. It eliminates hallucination and ensures answers are based on actual document content. Better chunking, embedding and search type leads to the better quality of the answers.

## Business Recommendations:
- Enhance the model by adding more features like "Extract Key Recommendations", "Summarize a sepcific section", "Compare insights from different sources"
- The model can used for knowledge/ research use cases like Business Analysis Report, internal knowledge bases, policies.
- Model can be improved by fine tuning the LLM parameters,chunking parameters and expermenting with other search types
- Model needs to be tracked and evaluated continously for better performance and relevance.  





<font size=6 color='#4682B4'>Power Ahead</font>
___